In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sqlite3
import scipy
import json
from estnltk import Text
from estnltk.taggers import VabamorfAnalyzer
import sys

sys.path.append('../../../common_code')

In [ ]:
from db_operations.db_display import *

## I Setup

In [16]:
SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
CONFLICT_DB = "syntax_morphology_conflicts.db"
SENTENCES_DB = "v33_koondkorpus_sentences_sentences_20250220-130121.db"
UNRESOLVED_IDX = "unresolved_idx.db"
RESULT_ONE_POSSIBLE_CASE = "one_possible_case.db"
RESULT_ADVMOD = "advmod.db"
RESULT_ADVMOD_DEPREL_CONFLICT = "advmod_deprel_conflict.db"
RESULT_NSUBJ_PART = "nsubj_part.db"
RESULT_FIRST_POSITION = "first_position.db"
RESULT_UPPERCASE = "uppercase.db"

In [17]:
morph_analyzer = VabamorfAnalyzer()

In [18]:
def number_of_field_elements(field: str) -> int:
    return len(json.loads(field).split())

def starts_with_uppercase(field: str) -> bool:
    return field.istitle()

def vm_assigns_unique_case(phrase_field: str, phrase_root_lemma: str) -> bool:
    phrase_txt = Text(str(phrase_field)).tag_layer("words")
    has_unique_case = False
    for idx, word in enumerate(phrase_txt.words):
        analysis = morph_analyzer.analyze_token(word.text)
        # huvipakkuv sõna leitakse tabelis oleva phrase_root_lemma alusel
        if len(analysis) == 1 and analysis[0]["lemma"] == phrase_root_lemma:
            has_unique_case = True
            break
    return has_unique_case

# seda praegu ei kasuta
def get_n_vabamorf_cases(phrase_field: str, phrase_root_lemma: str, n_cases: int) -> bool:
    form_mapping = {
      'sg p': 'part',
      'sg g': 'gen',
      'sg n': 'nom',
      'pl p': 'part',
      'pl g': 'gen',
      'pl n': 'nom',
      'sg el': 'el',
      'pl el': 'el',
      'sg tr': 'tr',
      'pl tr': 'tr',
      'sg in': 'in',
      'pl in': 'in',
      'sg es': 'es',
      'pl es': 'es',
      'sg ter': 'ter',
      'pl ter': 'ter',
      'sg kom': 'kom',
      'pl kom': 'kom',
      'sg ill': 'ill',
      'pl ill': 'ill',
      'sg abl': 'abl',
      'pl abl': 'abl',
      'sg ad': 'ad',
      'pl ad': 'ad',
      'sg all': 'all',
      'pl all': 'all',
      'adt':'adt',
      '':'',
      '?': '?'
  }
    phrase_txt = Text(str(phrase_field)).tag_layer("words")
    lemma_form = {}
    for idx, word in enumerate(phrase_txt.words):
        analysis = morph_analyzer.analyze_token(word.text)
        for idx2, a in enumerate(analysis):
            if a["lemma"] == phrase_root_lemma and a["partofspeech"] != "V":
                lemma_form[a["lemma"]] = []
    for idx, word in enumerate(phrase_txt.words):
        analysis = morph_analyzer.analyze_token(word.text)
        for idx2, a in enumerate(analysis):
            if a["lemma"] == phrase_root_lemma and a["partofspeech"] != "V":
                lemma_form[a["lemma"]]+=[form_mapping[a["form"]]]
    n_analyses = []
    for key, value in lemma_form.items():
        n_analyses.append(len(set(value)))
    return n_cases in n_analyses

## II Filtering of morpho-syntax conflicts

### I Conflicting word forms with with unique case

In [ ]:
# Esimesena juhud, mille kohta vabamorf ütlebki, et ainult üks analüüs võimalik
# Endiselt ei tea päris kindlalt, mis asi "possible_cases" väli on

con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

con.create_function("vm_assigns_unique_case", 2, vm_assigns_unique_case)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_ONE_POSSIBLE_CASE}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.one_possible_case
""")

cur.execute("""
CREATE TABLE result.one_possible_case 
AS
SELECT 
    tbl1.id,
    tbl1.pattern_id,
    tbl1.sentence_id,
    tbl1.verb_loc,
    tbl1.compound_loc,
    tbl1.phrase_root_loc,
    tbl1.verb_phrase_loc,
    tbl1.phrase_case,
    tbl1.phrase_deprel,
    tbl1.verb,
    tbl1.verb_compound,
    tbl1.phrase,
    tbl1.phrase_root_lemma,
    tbl1.current_analysis,
    tbl1.current_case,
    tbl1.possible_cases,
    tbl2.sentence
FROM
(
    SELECT
        *
    FROM
        syntax_morphology_conflicts
    WHERE 
        (verb, verb_compound) 
        IN 
        (
            SELECT 
                verb, 
                verb_compound
            FROM 
                syntax_morphology_conflicts
            GROUP BY 
                verb, verb_compound
            HAVING COUNT(DISTINCT phrase_case) = 1
        )
        AND
        vm_assigns_unique_case(phrase, phrase_root_lemma)
) AS tbl1
INNER JOIN
(
    SELECT
        id,
        text AS sentence
    FROM
        sents.sentences
) AS tbl2
ON
    tbl1.sentence_id = tbl2.id          
""")

cur.execute("""
DROP TABLE IF EXISTS idxs.unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.unresolved_idx
AS
SELECT
    tbl1.id AS idx
FROM
    syntax_morphology_conflicts AS tbl1
LEFT JOIN
    result.one_possible_case AS tbl2
ON 
    tbl1.id=tbl2.id
WHERE 
    tbl2.id IS NULL
""")


con.close()

In [42]:
display_sqlite_as_dataframe(f"{SOURCE_DATA_PATH}{RESULT_ONE_POSSIBLE_CASE}", 'one_possible_case', 5)

,id,pattern_id,sentence_id,verb_loc,compound_loc,phrase_root_loc,verb_phrase_loc,phrase_case,phrase_deprel,verb,verb_compound,phrase,phrase_root_lemma,current_analysis,current_case,possible_cases,sentence
0,1,1,30442,3,null,4,"""[1, 2, 3, 4]""",part,obl,aasima,,Ma siis aasisin Deani,Dea,"prop,sg,term",term,"""[null, \""nom\"", \""term\"", \""part\"", \""gen\"", \""adit\""]""","Ma siis aasisin Deani : kui sa Renatega voodisse lähed , kas tõmbad talle koti pähe ? ”"
1,5144,16,12776941,2,null,3,"""[1, 2, 3, 4, 6, 8]""",all,nsubj,ajama,,Maikuus ajas Nicole asja teravamaks väites,Nicole,"nom,prop,sg",nom,"""[\""all\""]""","Maikuus ajas Nicole asja veelgi teravamaks , väites , et oli rase , kui Tom ta maha jättis ."
2,5161,16,14171995,3,null,1,"""[1, 3, 5, 6, 8, 9]""",all,nsubj,ajama,,Nicole ajab armastuse ekstreemspordi vastu vanemate kaela,Nicole,"nom,prop,sg",nom,"""[\""all\""]""",Nicole Kidman ajab oma armastuse ekstreemspordi vastu vanemate kaela .
3,5189,16,19603461,12,null,10,"""[10, 11, 12, 13, 15, 19]""",all,obl,ajama,,KUULe ta ajab kyll juttu jummal,KUUL,"nominal,part,pl",part,"""[null, \""nom\"", \""all\"", \""part\""]""",pizike: sa oled teda Shadnäind üldse we ??? KUULe ta ajab kyll sellist juttu et ... no jummal
4,5197,16,21107371,25,null,18,"""[18, 20, 21, 22, 24, 25, 26, 27]""",all,obl,ajama,,keele suhtes h2tta j22n siis keelega ajab asja 2ra,keel,"com,gen,sg",gen,"""[\""all\"", \""gen\""]""","Ssidney: Magicfriend , kulla s6ber , idavirusse ple mul plaanis minna olnudki , ja kui vene keele suhtes h2tta j22n siis inglise keelega ajab asja 2ra"


### II Conflicting adverbials

In [ ]:
# Teiseks juhud, kus deprel on advmod/advcl
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

con.create_function("number_of_field_elements", 1, number_of_field_elements)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_ADVMOD}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.advmod
""")

cur.execute("""
CREATE TABLE result.advmod 
AS
SELECT 
    tbl1.id,
    tbl1.pattern_id,
    tbl1.sentence_id,
    tbl1.verb_loc,
    tbl1.compound_loc,
    tbl1.phrase_root_loc,
    tbl1.verb_phrase_loc,
    tbl1.phrase_case,
    tbl1.phrase_deprel,
    tbl1.verb,
    tbl1.verb_compound,
    tbl1.phrase,
    tbl1.phrase_root_lemma,
    tbl1.current_analysis,
    tbl1.current_case,
    tbl1.possible_cases,
    tbl2.sentence
FROM
(
    SELECT
        *
    FROM
        syntax_morphology_conflicts AS conf
    INNER JOIN
        idxs.unresolved_idx AS tmp
    ON
        conf.id=tmp.idx
    WHERE
        phrase_deprel="advmod" OR phrase_deprel="advcl"
        AND 
        number_of_field_elements(conf.possible_cases) = 2
        AND 
        instr(conf.possible_cases, "null") > 0
) AS tbl1
INNER JOIN
(
    SELECT
        id,
        text AS sentence
    FROM
        sents.sentences
) AS tbl2
ON
    tbl1.sentence_id = tbl2.id          
""")


cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.advmod AS tbl2
ON 
    tbl1.idx=tbl2.id
WHERE 
    tbl2.id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [41]:
display_sqlite_as_dataframe(f"{SOURCE_DATA_PATH}{RESULT_ADVMOD}", 'advmod', 5)

,id,pattern_id,sentence_id,verb_loc,compound_loc,phrase_root_loc,verb_phrase_loc,phrase_case,phrase_deprel,verb,verb_compound,phrase,phrase_root_lemma,current_analysis,current_case,possible_cases,sentence
0,24179,1045,914890,17,"""[15]""",11,"""[11, 13, 14, 15, 16, 17]""",abl,advmod,astuma,läbi,sealt enne kevadet keegi läbi ei astu,sealt,,,"""[null, \""abl\""]""","Tema sõnul ei tohi suvilat nõnda maha jätta , et sealt enne kevadet keegi läbi ei astu ."
1,24180,1045,1113115,21,"""[22]""",20,"""[20, 21, 22, 25, 26]""",abl,advmod,astuma,läbi,sealt astub läbi seltskond pillimehi,sealt,,,"""[null, \""abl\""]""","Cuscos ja Punos on rahvamuusikud peaaegu painav nähtus : vähegi korralikumas restoranis õhtust süües võib kindel olla , et sealt astub läbi vähemalt üks seltskond pillimehi ( halvemal juhul pillimeheks pürgijaid ) , kes paar lugu esitavad ja selle eest loomulikult ka tasu nõuavad ."
2,24181,1045,1594227,4,"""[6]""",5,"""[4, 5, 6]""",abl,advmod,astuma,läbi,astun sealt läbi,sealt,,,"""[null, \""abl\""]""",Praegu mõtlen : astun sealt läbi .
3,24182,1045,2304422,1,"""[4]""",3,"""[1, 2, 3, 4]""",abl,advmod,astuma,läbi,Astusin eile sealt läbi,sealt,,,"""[null, \""abl\""]""","Astusin eile sealt läbi ja kogesin taas , et selles poes on tore käia ."
4,24183,1045,3239451,25,"""[24]""",21,"""[16, 18, 20, 21, 23, 24, 25]""",abl,advmod,astuma,läbi,kord aasta tagant mitte lihtsalt õhtuks läbi astuvad,lihtsalt,,,"""[null, \""abl\""]""","Euroopa meistrivõistlused jalgpallis on nagu kauged sugulased , kes harva külas käivad , aga kui kord nelja aasta tagant mitte lihtsalt üheks õhtuks läbi astuvad , vaid ennast kolmeks nädalaks mugavalt sisse seavad , siis on terveks selleks ajaks paras segadus majas ."


In [ ]:
# Kolmandaks juhud, kus juursõna käitub kui määrus, aga deprel pole advmod/advcl
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

con.create_function("number_of_field_elements", 1, number_of_field_elements)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_ADVMOD_DEPREL_CONFLICT}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.advmod_wrong_deprel
""")

cur.execute("""
CREATE TABLE result.advmod_wrong_deprel 
AS
SELECT 
    tbl1.id,
    tbl1.pattern_id,
    tbl1.sentence_id,
    tbl1.verb_loc,
    tbl1.compound_loc,
    tbl1.phrase_root_loc,
    tbl1.verb_phrase_loc,
    tbl1.phrase_case,
    tbl1.phrase_deprel,
    tbl1.verb,
    tbl1.verb_compound,
    tbl1.phrase,
    tbl1.phrase_root_lemma,
    tbl1.current_analysis,
    tbl1.current_case,
    tbl1.possible_cases,
    tbl2.sentence
FROM
(
    SELECT
        *
    FROM
        syntax_morphology_conflicts AS conf
    INNER JOIN
        idxs.unresolved_idx AS tmp
    ON
        conf.id=tmp.idx
    WHERE
        number_of_field_elements(conf.possible_cases) = 2
        AND 
        instr(conf.possible_cases, "null") > 0
) AS tbl1
INNER JOIN
(
    SELECT
        id,
        text AS sentence
    FROM
        sents.sentences
) AS tbl2
ON
    tbl1.sentence_id = tbl2.id          
""")


cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.advmod_wrong_deprel AS tbl2
ON 
    tbl1.idx=tbl2.id
WHERE 
    tbl2.id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [40]:
display_sqlite_as_dataframe(f"{SOURCE_DATA_PATH}{RESULT_ADVMOD_DEPREL_CONFLICT}", 'advmod_wrong_deprel', 5)

,id,pattern_id,sentence_id,verb_loc,compound_loc,phrase_root_loc,verb_phrase_loc,phrase_case,phrase_deprel,verb,verb_compound,phrase,phrase_root_lemma,current_analysis,current_case,possible_cases,sentence
0,76423,1051,3091351,33,"""[37]""",32,"""[30, 32, 33, 36, 37, 40]""",abl,obl,jooksma,läbi,Komisjoni vahelt jookseb küsimustes läbi lõhet,vahelt,post,,"""[null, \""abl\""]""","Kogunemisest kujuneb ilmselt üks viimaste aastate vastuolulisemaid , kuna üheksa osalise - USA , Kanada , Jaapani , Saksamaa , Suurbritannia , Prantsusmaa , Itaalia , Venemaa ja Euroopa Komisjoni - vahelt jookseb kõige tähtsamates küsimustes läbi mitu sügavat lõhet ."
1,160396,1055,2223480,14,"""[13]""",12,"""[10, 11, 12, 13, 14, 19]""",abl,obl,käima,läbi,inimesed sealt vahelt läbi käivad suuda,vahelt,post,,"""[null, \""abl\""]""",""" Meil ei ole võimalik nii ehitada , et inimesed sealt vahelt läbi käivad , sest me ei suuda garanteerida neile ohutut liikumist ."
2,169669,947,5884965,24,null,18,"""[18, 20, 21, 22, 23, 24, 26]""",abl,obl,küsima,,Omalt poolt ei ole ma iial küsinud härra,Omalt,"nom,prop,sg",nom,"""[null, \""abl\""]""","Poliitikute stagneerunud mõtteviis ärritas Andres Didod , ta kirjutas 1918. aasta 11. septembril saadik Thomas'le : "" Omalt poolt ei ole ma iial küsinud ei härra ministrilt ega komiteelt muud kui seda , mida meie diplomaatilised esindajad ei ole väsinud küsimast kõigilt valitsustelt : Eesti iseseisvuse tunnustamist ja kindlustust , et liitlaste valitsused ei taha uuesti meie maad viia Vene ülemvõimu alla või aidata teda ära võtta . """
3,172537,965,969797,8,null,12,"""[5, 8, 10, 12, 14]""",abl,obl,lahkuma,,peatoimetaja lahkub aprillil kohalt andes,kohalt,post,,"""[null, \""abl\""]""","Eesti suurima Päevalehe Postimees peatoimetaja Marko Mihkelson lahkub 17. aprillil oma kohalt , andes lehe juhtimise üle oma senisele asetäitjale Urmas Klaasile ."
4,172572,965,1329678,8,null,10,"""[8, 10, 11]""",abl,obl,lahkuma,,lahkusin kohalt valitsuses,kohalt,post,,"""[null, \""abl\""]""",Sellega oli minu töö jällegi tehtud ja lahkusin oma kohalt valitsuses .


### III Conflicting subjects

In [ ]:
# Neljandaks juhud, kus deprel nsubj ja võimalike käänete hulgas part
# uuesti Hendriku tööd vaadates tundub, et ikkagi possible_cases seest tuleb otsida, mitte rektsioonil põhinevat käänet vaadata
# KÜSIMUS: Kas loogika on, et kuigi ühestamisel saab õige käände (nom), siis võimalike variantide hulgas partitiivi nähes
# teame, et tegemist on sõnadega, milles esineb üleüldiselt veaohtlik vormihomonüümia?
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_NSUBJ_PART}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.nsubj_part
""")

cur.execute("""
CREATE TABLE result.nsubj_part 
AS
SELECT 
    tbl1.id,
    tbl1.pattern_id,
    tbl1.sentence_id,
    tbl1.verb_loc,
    tbl1.compound_loc,
    tbl1.phrase_root_loc,
    tbl1.verb_phrase_loc,
    tbl1.phrase_case,
    tbl1.phrase_deprel,
    tbl1.verb,
    tbl1.verb_compound,
    tbl1.phrase,
    tbl1.phrase_root_lemma,
    tbl1.current_analysis,
    tbl1.current_case,
    tbl1.possible_cases,
    tbl2.sentence
FROM
(
    SELECT
        *
    FROM
        syntax_morphology_conflicts AS conf
    INNER JOIN
        idxs.unresolved_idx AS tmp
    ON
        conf.id=tmp.idx
    WHERE
        phrase_deprel = "nsubj"
        AND
        instr(possible_cases, "part") > 0
) AS tbl1
INNER JOIN
(
    SELECT
        id,
        text AS sentence
    FROM
        sents.sentences
) AS tbl2
ON
    tbl1.sentence_id = tbl2.id          
""")

cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.nsubj_part AS tbl2
ON 
    tbl1.idx=tbl2.id
WHERE 
    tbl2.id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [39]:
display_sqlite_as_dataframe(f"{SOURCE_DATA_PATH}{RESULT_NSUBJ_PART}", 'nsubj_part', 5)

,id,pattern_id,sentence_id,verb_loc,compound_loc,phrase_root_loc,verb_phrase_loc,phrase_case,phrase_deprel,verb,verb_compound,phrase,phrase_root_lemma,current_analysis,current_case,possible_cases,sentence
0,3,1,1081322,8,null,9,"""[8, 9]""",part,nsubj,aasima,,aasib Taavi,Taavi,"nom,prop,sg",nom,"""[null, \""nom\"", \""part\"", \""gen\"", \""adit\""]""",""" Aga kujunes veidi teisiti , "" aasib Taavi Teplenkov ."
1,8,1,5374790,16,null,17,"""[16, 17]""",part,nsubj,aasima,,aasib Jüri,Jüri,"nom,prop,sg",nom,"""[null, \""nom\"", \""part\"", \""gen\"", \""adit\""]""",""" Kuule , sina , kas sa vahel oma rekvisiidi koju ka jätad ? "" aasib Jüri Vlassov Peeter Volkonski kallal , kes hallile plastkargule toetudes üle näiteplatsi lonkab ."
2,9,1,7384132,6,null,7,"""[6, 7]""",part,nsubj,aasima,,aasis Aave,Aave,"nom,prop,sg",nom,"""[\""part\"", \""nom\"", \""gen\""]""","“ Keeled suus , ” aasis Aave ja sirutas oma punase keele võõbatud huulte vahelt kaugele ette ."
3,12,1,12950974,16,null,17,"""[16, 17]""",part,nsubj,aasima,,aasib Jüri,Jüri,"nom,prop,sg",nom,"""[null, \""nom\"", \""part\"", \""gen\"", \""adit\""]""",""" Kuule , sina , kas sa vahel oma rekvisiidi koju ka jätad ? "" aasib Jüri Vlassov Peeter Volkonski kallal , kes hallile plastkargule toetudes üle näiteplatsi lonkab ."
4,13,1,15638719,6,null,7,"""[6, 7]""",part,nsubj,aasima,,aasis õde,õde,"com,nom,sg",nom,"""[\""part\"", \""nom\""]""","Kutsume teda professoriks , » aasis õde Kristina ."


### IV Conflicting word forms that are probably subjects

In [ ]:
# Viiendaks juhud, kus juursõnaks fraasi esimene sõna, on eeldus, et valdavalt peaks see olema nsubj
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_FIRST_POSITION}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.first_position
""")

cur.execute("""
CREATE TABLE result.first_position 
AS
SELECT 
    tbl1.id,
    tbl1.pattern_id,
    tbl1.sentence_id,
    tbl1.verb_loc,
    tbl1.compound_loc,
    tbl1.phrase_root_loc,
    tbl1.verb_phrase_loc,
    tbl1.phrase_case,
    tbl1.phrase_deprel,
    tbl1.verb,
    tbl1.verb_compound,
    tbl1.phrase,
    tbl1.phrase_root_lemma,
    tbl1.current_analysis,
    tbl1.current_case,
    tbl1.possible_cases,
    tbl2.sentence
FROM
(
    SELECT
        *
    FROM
        syntax_morphology_conflicts AS conf
    INNER JOIN
        idxs.unresolved_idx AS tmp
    ON
        conf.id = tmp.idx
    WHERE
        phrase_root_loc=1
) AS tbl1
INNER JOIN
(
    SELECT
        id,
        text AS sentence
    FROM
        sents.sentences
) AS tbl2
ON
    tbl1.sentence_id = tbl2.id          
""")

cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.first_position AS tbl2
ON 
    tbl1.idx=tbl2.id
WHERE 
    tbl2.id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [38]:
display_sqlite_as_dataframe(f"{SOURCE_DATA_PATH}{RESULT_FIRST_POSITION}", 'first_position', 5)

,id,pattern_id,sentence_id,verb_loc,compound_loc,phrase_root_loc,verb_phrase_loc,phrase_case,phrase_deprel,verb,verb_compound,phrase,phrase_root_lemma,current_analysis,current_case,possible_cases,sentence
0,76430,1051,6986872,2,"""[3]""",1,"""[1, 2, 3, 5]""",abl,obl,jooksma,läbi,Krundilt jookseb läbi oja,Krundilt,"nom,prop,sg",nom,"""[\""abl\""]""","Krundilt jookseb läbi Mähe oja , mille äärde kinnisvarafirmal on plaanis ehitada kuni nelja boksiga ridaelamuid või paarismaju ."
1,137810,2389,9349831,5,"""[4]""",1,"""[1, 2, 3, 4, 5]""",abl,nsubj,kolima,ära,Wiiralt oli Stockholmist ära kolinud,Wiiralt,"nom,prop,sg",nom,"""[null, \""nom\"", \""abl\""]""","Wiiralt oli Stockholmist ära kolinud , talle ei meeldinud Rootsi ."
2,160399,1055,2485005,2,"""[7]""",1,"""[1, 2, 4, 6, 7, 8]""",abl,obl,käima,läbi,Peolt käis sõnul korraks läbi ärimees,Peolt,"nom,prop,sg",nom,"""[\""abl\""]""",Peolt käis enda sõnul vaid korraks läbi ärimees Rein Kilk .
3,160427,1055,3640801,2,"""[5]""",1,"""[1, 2, 4, 5, 9]""",abl,obl,käima,läbi,Peolt käib rõõmuks läbi päkapikk,Peolt,"nom,prop,sg",nom,"""[\""abl\""]""",Peolt käib laste rõõmuks läbi ka üks unine päkapikk .
4,170163,947,7871244,4,null,1,"""[1, 2, 3, 4]""",abl,obj,küsima,,Rahvalt keegi ei küsi,Rahvalt,"nom,prop,sg",nom,"""[\""abl\""]""","Rahvalt keegi ei küsi , aga siiamaani on küsitlused näidanud , et soomlased marka ERMiga siduda ei taha , ehk ainult juhul , kui Rootsi kroon liidetakse ."


### V Conflicting proper names

In [ ]:
# Kuuendaks pärisnimed
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

con.create_function("starts_with_uppercase", 1, starts_with_uppercase)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_UPPERCASE}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.uppercase
""")

cur.execute("""
CREATE TABLE result.uppercase 
AS
SELECT 
    tbl1.id,
    tbl1.pattern_id,
    tbl1.sentence_id,
    tbl1.verb_loc,
    tbl1.compound_loc,
    tbl1.phrase_root_loc,
    tbl1.verb_phrase_loc,
    tbl1.phrase_case,
    tbl1.phrase_deprel,
    tbl1.verb,
    tbl1.verb_compound,
    tbl1.phrase,
    tbl1.phrase_root_lemma,
    tbl1.current_analysis,
    tbl1.current_case,
    tbl1.possible_cases,
    tbl2.sentence
FROM
(
    SELECT
        *
    FROM
        syntax_morphology_conflicts AS conf
    INNER JOIN
        idxs.unresolved_idx AS tmp
    ON
        conf.id = tmp.idx
    WHERE
        starts_with_uppercase(phrase_root_lemma)
) AS tbl1
INNER JOIN
(
    SELECT
        id,
        text AS sentence
    FROM
        sents.sentences
) AS tbl2
ON
    tbl1.sentence_id = tbl2.id          
            """)

cur.execute("""
DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
CREATE TABLE idxs.new_unresolved_idx
AS
SELECT
    tbl1.idx
FROM
    idxs.unresolved_idx AS tbl1
LEFT JOIN
    result.uppercase AS tbl2
ON 
    tbl1.idx=tbl2.id
WHERE 
    tbl2.id IS NULL
""")

cur.execute("""
DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [37]:
display_sqlite_as_dataframe(f"{SOURCE_DATA_PATH}{RESULT_UPPERCASE}", 'uppercase', 5)

,id,pattern_id,sentence_id,verb_loc,compound_loc,phrase_root_loc,verb_phrase_loc,phrase_case,phrase_deprel,verb,verb_compound,phrase,phrase_root_lemma,current_analysis,current_case,possible_cases,sentence
0,76884,2287,1698843,14,"""[19]""",15,"""[10, 14, 15, 17, 18, 19]""",abl,nsubj,jooksma,välja,tsenderduste ajal jooksis Hoult korral ebakindlalt välja,Hoult,"nom,prop,sg",nom,"""[null, \""nom\"", \""abl\""]""","Seda kinnitas eestlasest puurivaht küll , et teise poolaja tsenderduste ja nurgalöökide ajal jooksis Hoult paaril korral ebakindlalt välja ."
1,137814,2389,10930785,13,"""[19]""",18,"""[13, 17, 18, 19]""",abl,advcl,kolima,ära,kolis südamega Injult ära,Injult,"nom,prop,sg",nom,"""[null, \""nom\"", \""abl\""]""","Võttis oma kaks hobust ja kaks lehma , asetas linnupuuri vankrile ning kolis nagu mustlane murtud südamega Injult ära ."
2,144248,791,2678355,2,null,3,"""[1, 2, 3]""",abl,obl,kukkuma,,TASSID KUKUVAD LAUALT,Laualt,"nom,prop,sg",nom,"""[\""abl\""]""",TASSID KUKUVAD LAUALT ALLA .
3,144400,791,5124263,3,null,4,"""[1, 3, 4]""",abl,obl,kukkuma,,LAPSE KUKKUS SEINALT,Seinalt,"nom,prop,sg",nom,"""[\""abl\""]""",LAPSE VOODI KUKKUS SEINALT ALLA .
4,144721,791,11117228,24,null,23,"""[23, 24, 25]""",abl,nsubj,kukkuma,,Seintelt kukkus krohvi,Seintelt,"nom,prop,sg",nom,"""[\""abl\""]""","Kohapeal pidi ta aga valmistulemuse silme ette manamiseks võtma appi kogu oma fantaasialennu maja oli ikka sedavõrd lääbakil ja räämas : « Seintelt kukkus krohvi , kogu hoone ja ümbrus oli traate ja muud kola äärest ääreni täis . »"
